<a href="https://colab.research.google.com/github/krishnasaijoga/OpenEnvHackathon/blob/main/Mod_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Clone, Modify, Deploy

Clone an existing environment, modify it, test locally, and deploy to HF Spaces.

**Time:** ~25 min · **Difficulty:** Intermediate · **GPU:** Not required

> **Note:** The deployment steps (`openenv push`) require a Hugging Face account and token. The local testing steps work without one.

In [1]:
# Install dependencies
!uv pip install -q openenv-core==0.2.2 "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/echo_env"

## 1. Verify the Hosted Echo Environment

First, let's confirm the hosted Echo environment works.

In [2]:
from echo_env import EchoEnv, CallToolAction

ECHO_URL = "https://mroyme-echo-env.hf.space"

with EchoEnv(base_url=ECHO_URL).sync() as env:
    result = env.reset()
    result = env.step(CallToolAction(tool_name="echo_with_length", arguments= {"message": "Hello, OpenEnv!"}))
    print(f"Sent: ping")
    print(f"Received: {result.observation}")
    print(f"This is the standard Echo — it returns exactly what you send.")

Sent: ping
Received: done=False reward=None metadata={} tool_name='echo_with_length' result={'content': [{'type': 'text', 'text': '{"message":"Hello, OpenEnv!","length":15}', 'annotations': None, 'meta': None}], 'structured_content': {'message': 'Hello, OpenEnv!', 'length': 15}, 'meta': None, 'data': {'message': 'Hello, OpenEnv!', 'length': 15}, 'is_error': False} error=None
This is the standard Echo — it returns exactly what you send.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Clone the Echo Environment

Clone the Space repository to get the full source code.

In [3]:
!git clone https://huggingface.co/spaces/mroyme/echo_env echo-env-modified 2>/dev/null || echo "Already cloned"

import os
os.listdir("echo-env-modified")

['.git',
 'Dockerfile',
 'openenv.yaml',
 'uv.toml',
 'README.md',
 '.gitattributes',
 '__init__.py',
 'server',
 'client.py',
 'openenv_echo_env.egg-info',
 'envs',
 'src',
 'pyproject.toml',
 'wheels']

## 3. Explore the Structure

Every OpenEnv environment follows the same layout.

In [4]:
!find echo-env-modified -type f -name "*.py" -o -name "*.yaml" -o -name "*.toml" -o -name "Dockerfile" | sort

echo-env-modified/client.py
echo-env-modified/Dockerfile
echo-env-modified/envs/echo_env/client.py
echo-env-modified/envs/echo_env/__init__.py
echo-env-modified/envs/echo_env/openenv.yaml
echo-env-modified/envs/echo_env/pyproject.toml
echo-env-modified/envs/echo_env/server/app.py
echo-env-modified/envs/echo_env/server/Dockerfile
echo-env-modified/envs/echo_env/server/echo_environment.py
echo-env-modified/envs/echo_env/server/__init__.py
echo-env-modified/envs/echo_env/uv.toml
echo-env-modified/__init__.py
echo-env-modified/openenv.yaml
echo-env-modified/pyproject.toml
echo-env-modified/server/app.py
echo-env-modified/server/Dockerfile
echo-env-modified/server/echo_environment.py
echo-env-modified/server/__init__.py
echo-env-modified/src/core/client_types.py
echo-env-modified/src/core/containers/images/Dockerfile
echo-env-modified/src/core/containers/__init__.py
echo-env-modified/src/core/containers/runtime/daytona_provider.py
echo-env-modified/src/core/containers/runtime/__init__.py
ec

In [5]:
# Look at the environment logic
!cat echo-env-modified/envs/echo_env/server/echo_environment.py

# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Echo Environment Implementation.

A pure MCP environment that echoes back messages sent to it.
This demonstrates how to build an MCPEnvironment with inline FastMCP tools.

All interactions happen through MCP tools:
- `echo_message(message)`: Echo back the provided message
- `echo_with_length(message)`: Echo back the message with its length

Example:
    >>> from openenv.core.env_server.mcp_types import ListToolsAction, CallToolAction
    >>> env = EchoEnvironment()
    >>> env.reset()
    >>>
    >>> # List available tools
    >>> obs = env.step(ListToolsAction())
    >>> print([t.name for t in obs.tools])  # ["echo_message", "echo_with_length"]
    >>>
    >>> # Call a tool
    >>> obs = env.step(CallToolAction(tool_name="echo_message", arguments={"message": "Hi!"}))
    >>> 

## 4. Modify the Environment

Let's make a "Reverse Echo" — instead of echoing back the message, it reverses it.

We'll modify the `step()` method in `environment.py`.

In [6]:
# Read the current environment.py
env_file = "echo-env-modified/envs/echo_env/server/echo_environment.py"

with open(env_file, "r") as f:
    content = f.read()

print("Original environment.py:")
print(content)

Original environment.py:
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Echo Environment Implementation.

A pure MCP environment that echoes back messages sent to it.
This demonstrates how to build an MCPEnvironment with inline FastMCP tools.

All interactions happen through MCP tools:
- `echo_message(message)`: Echo back the provided message
- `echo_with_length(message)`: Echo back the message with its length

Example:
    >>> from openenv.core.env_server.mcp_types import ListToolsAction, CallToolAction
    >>> env = EchoEnvironment()
    >>> env.reset()
    >>>
    >>> # List available tools
    >>> obs = env.step(ListToolsAction())
    >>> print([t.name for t in obs.tools])  # ["echo_message", "echo_with_length"]
    >>>
    >>> # Call a tool
    >>> obs = env.step(CallToolAction(tool_name="echo_message", arguments={"me

In [11]:
# Read the current environment.py
env_file = "echo-env-modified/envs/echo_env/server/echo_environment.py"

with open(env_file, "r") as f:
    content = f.read()

# Modify: reverse the echo message in the tool functions
# This time, target the 'message' variable directly within the tool definitions.
modified = content.replace(
    '            return message', # Original line in echo_message
    '            return message[::-1]', # Modified line
    1 # Replace only the first occurrence
)

modified = modified.replace(
    '            return {"message": message, "length": len(message)}', # Original line in echo_with_length
    '            return {"message": message[::-1], "length": len(message)}', # Modified line
    1 # Replace only the first occurrence
)

with open(env_file, "w") as f:
    f.write(modified)

print("Modified environment.py (now correctly reverses the message in tool functions):")
print(modified)

Modified environment.py (now correctly reverses the message in tool functions):
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

"""
Echo Environment Implementation.

A pure MCP environment that echoes back messages sent to it.
This demonstrates how to build an MCPEnvironment with inline FastMCP tools.

All interactions happen through MCP tools:
- `echo_message(message)`: Echo back the provided message
- `echo_with_length(message)`: Echo back the message with its length

Example:
    >>> from openenv.core.env_server.mcp_types import ListToolsAction, CallToolAction
    >>> env = EchoEnvironment()
    >>> env.reset()
    >>>
    >>> # List available tools
    >>> obs = env.step(ListToolsAction())
    >>> print([t.name for t in obs.tools])  # ["echo_message", "echo_with_length"]
    >>>
    >>> # Call a tool
    >>> obs = env.step(

## 5. Test Locally

Start the modified server and connect to it.

> In Colab, we'll start the server as a background process. Locally, you'd run `uv run server` in a separate terminal.

In [12]:
import subprocess
import time
import sys

# Start the server in the background
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "echo_env.server.app:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd="echo-env-modified",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give it time to start
time.sleep(3)
print(f"Server started (PID: {server.pid})")

Server started (PID: 3004)


In [13]:
import time

# Test the modified environment
MAX_RETRIES = 10
RETRY_DELAY = 2 # seconds

for attempt in range(MAX_RETRIES):
    try:
        with EchoEnv(base_url="http://localhost:8000").sync() as env:
            result = env.reset()

            test_messages = ["Hello", "OpenEnv", "Reverse this!"]
            for msg in test_messages:
                result = env.step(CallToolAction(tool_name="echo_with_length", arguments= {"message": msg}))
                print(f"Sent: {msg:20s} \u2192 Received: {result.observation}")
            print(f"Successfully connected and tested after {attempt + 1} attempts.")
            break # Exit loop if successful
    except ConnectionError as e:
        print(f"Attempt {attempt + 1}/{MAX_RETRIES}: Failed to connect to server ({e}). Retrying in {RETRY_DELAY} seconds...")
        time.sleep(RETRY_DELAY)
else:
    print(f"Failed to connect to the server after {MAX_RETRIES} attempts. Please ensure the server in the previous cell is running correctly.")

Sent: Hello                → Received: done=False reward=None metadata={} tool_name='echo_with_length' result=None error=ToolError(error_type=<ToolErrorType.TIMEOUT: 'timeout'>, message="Tool 'echo_with_length' timed out after 30.0 seconds")
Sent: OpenEnv              → Received: done=False reward=None metadata={} tool_name='echo_with_length' result=None error=ToolError(error_type=<ToolErrorType.TIMEOUT: 'timeout'>, message="Tool 'echo_with_length' timed out after 30.0 seconds")
Sent: Reverse this!        → Received: done=False reward=None metadata={} tool_name='echo_with_length' result=None error=ToolError(error_type=<ToolErrorType.TIMEOUT: 'timeout'>, message="Tool 'echo_with_length' timed out after 30.0 seconds")
Successfully connected and tested after 1 attempts.


In [14]:
# Clean up the server
server.terminate()
server.wait()
print("Server stopped.")

Server stopped.


## 6. Deploy to HF Spaces

Once your environment works locally, deploy it with `openenv push`.

```bash
cd echo-env-modified
openenv push --repo-id YOUR_USERNAME/reverse-echo-env
```

Your environment is now live at:
- **API:** `https://YOUR_USERNAME-reverse-echo-env.hf.space`
- **Web UI:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/web`
- **Docs:** `https://YOUR_USERNAME-reverse-echo-env.hf.space/docs`

In [16]:
# Uncomment and run to deploy (requires HF token)
!cd echo-env-modified && openenv push --repo-id KrishnaJoga/reverse-echo-env

Validating OpenEnv environment in /content/echo-env-modified...
Usage: openenv push [OPTIONS] [DIRECTORY]
Try 'openenv push --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Invalid value: Invalid OpenEnv environment structure: Required file missing: │
│ models.py                                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯


## 7. Connect to Your Deployed Environment

After deployment, install the client and connect:

In [ ]:
# Uncomment after deploying
# !pip install -q git+https://huggingface.co/spaces/YOUR_USERNAME/reverse-echo-env
#
# with EchoEnv(base_url="https://YOUR_USERNAME-reverse-echo-env.hf.space").sync() as env:
#     result = env.reset()
#     result = env.step(EchoAction(message="Deployed!"))
#     print(f"Response from your Space: {result.observation}")

## 8. Docker Deployment (Alternative)

You can also pull and run the Docker image locally:

```bash
# Pull from HF registry (after deploying)
docker pull registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest
docker run -d -p 8001:8000 registry.hf.space/YOUR_USERNAME-reverse-echo-env:latest

# Or build from source
cd echo-env-modified
docker build -t reverse-echo:latest -f server/Dockerfile .
docker run -d -p 8001:8000 reverse-echo:latest
```

Connect the same way:
```python
with EchoEnv(base_url="http://localhost:8001").sync() as env:
    result = env.reset()
```

## Summary

What you did:
1. Cloned an existing environment from HF Spaces
2. Explored its structure (models, client, server)
3. Modified the environment logic (echo → reverse echo)
4. Tested locally with uvicorn
5. Deployed to HF Spaces with `openenv push`

The workflow is always: **clone → modify → test → deploy**.

**Next:** [Module 4](../module-4/README.md) — Building an environment from scratch.